In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

ROOT = Path('~/Downloads').expanduser()


def load_points(ply_path, n=5000):
    mesh = o3d.io.read_triangle_mesh(str(ply_path))
    if mesh.has_triangles():
        o3d.utility.random.seed(0)
        return np.asarray(mesh.sample_points_uniformly(number_of_points=n).points)
    return np.asarray(o3d.io.read_point_cloud(str(ply_path)).points)


def show(dataset, obj_ids=None, cols=4):
    """Mesh points, 32 keypoints and 12 edges of <dataset>/cached_keypoints_3d_edge.json (original BOP frame)."""
    folder = ROOT / dataset
    cache = json.loads((folder / 'cached_keypoints_3d_edge.json').read_text())
    audit = cache.get('metadata', {}).get('object_audit', {})
    p_path = folder / 'p_values.json'
    p_values = json.loads(p_path.read_text()) if p_path.exists() else {}
    obj_ids = obj_ids or sorted(int(k) for k in cache['keypoints_3d'])
    rows = (len(obj_ids) + cols - 1) // cols
    fig = plt.figure(figsize=(5 * cols, 5 * rows))
    for i, obj_id in enumerate(obj_ids):
        ax = fig.add_subplot(rows, cols, i + 1, projection='3d')
        points = load_points(folder / 'models_eval' / f'obj_{obj_id:06d}.ply')
        kpts = np.asarray(cache['keypoints_3d'][str(obj_id)])
        ax.scatter(*points.T, c='tab:blue', s=1, alpha=0.2)
        ax.scatter(*kpts[:16].T, c='red', s=30, edgecolors='darkred')
        ax.scatter(*kpts[16:].T, c='cyan', s=30, marker='s', edgecolors='darkcyan')
        for edge in cache['edges_mask'][str(obj_id)]:
            ax.plot(*kpts[edge].T, color='red', linewidth=1.5, alpha=0.7)
        span = np.ptp(np.vstack([points, kpts]), axis=0)
        ax.set_box_aspect(np.maximum(span / span.max(), 0.3))
        info = audit.get(str(obj_id), {})
        title = f'obj {obj_id}'
        if info:
            title += f"\n{info.get('box_frame', '?')} box, sym err {info.get('symmetry_error_mm', 0):.1f} mm"
        if any(p_values.get(str(obj_id), [0, 0, 0])):
            title += f"\nP = {np.round(p_values[str(obj_id)], 2).tolist()}"
        ax.set_title(title, fontsize=10)
    fig.suptitle(dataset, fontsize=16, fontweight='bold')
    fig.tight_layout()
    plt.show()


In [ ]:
show('lmo')

In [ ]:
show('tudl')

In [ ]:
show('icbin')

In [ ]:
show('itodd')

In [ ]:
show('ycbv')

In [ ]:
show('tless')

In [ ]:
show('hb')